# Mini-TP 2 — Metadatos del modelo por GraphQL (Sesión 2)

Expone los metadatos de **mi modelo** (el `iris-rf` del Mini-TP 1) por
**GraphQL** (FastAPI + Strawberry) y **compara** la misma lectura contra REST:
cuántas llamadas y cuántos bytes cuesta armar *la vista del modelo*
(`name`, `version`, `algorithm`, `auc`, `accuracy`, `f1`) con cada enfoque.

Los metadatos salen del artefacto entrenado (`model/metadata.json`), leídos por
`src/model_service/model.py`; las dos APIs (`rest.py`, `graphql.py`) comparten
ese mismo módulo.

## 1. Los metadatos reales de mi modelo

In [1]:
import json
meta = json.load(open("../model/metadata.json"))
meta

{'model_name': 'iris-rf',
 'model_version': '1.0.0',
 'algorithm': 'RandomForestClassifier',
 'dataset': 'sklearn.datasets.load_iris',
 'features': ['sepal_length_cm',
  'sepal_width_cm',
  'petal_length_cm',
  'petal_width_cm'],
 'target_names': ['setosa', 'versicolor', 'virginica'],
 'test_accuracy': 0.9,
 'test_f1': 0.8997,
 'test_auc': 0.9933,
 'trained_at': '2026-09-04T00:36:03.178930+00:00'}

## 2. Levantar las dos APIs en segundo plano

In [2]:
import threading, time, socket, uvicorn

def serve(app, port):
    def _run():
        uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port,
                                      log_level="warning")).run()
    threading.Thread(target=_run, daemon=True).start()
    for _ in range(50):
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                return
        except OSError:
            time.sleep(0.2)
    raise RuntimeError(f"el servidor en :{port} no respondió")

from model_service.rest import app as rest_app
from model_service.graphql import app as gql_app

serve(rest_app, 8100)
serve(gql_app, 8110)
REST = "http://127.0.0.1:8100"
GQL = "http://127.0.0.1:8110/graphql"
print("REST   ->", REST)
print("GraphQL->", GQL, "(GraphiQL en el navegador)")

REST   -> http://127.0.0.1:8100
GraphQL-> http://127.0.0.1:8110/graphql (GraphiQL en el navegador)


## 3. El esquema GraphQL

Definido en `src/model_service/graphql.py`:

```graphql
type Metrics { auc: Float!  accuracy: Float!  f1: Float! }
type Model   { name: String!  version: String!  algorithm: String!  metrics: Metrics! }
type Query   { model: Model! }
```

## 4. La query — una sola llamada, sólo los campos pedidos

In [3]:
import requests
QUERY = "{ model { name version algorithm metrics { auc accuracy f1 } } }"
rg = requests.post(GQL, json={"query": QUERY})
print("HTTP", rg.status_code)
rg.json()

HTTP 200


{'data': {'model': {'name': 'iris-rf',
   'version': '1.0.0',
   'algorithm': 'RandomForestClassifier',
   'metrics': {'auc': 0.9933, 'accuracy': 0.9, 'f1': 0.8997}}}}

## 5. Cliente Python: armar la misma vista por REST

In [4]:
# REST no tiene un endpoint "vista del modelo": hay que componer 2 llamadas
r_info = requests.get(f"{REST}/v1/model")             # name, version, algorithm (+ campos de más)
r_metrics = requests.get(f"{REST}/v1/model/metrics")  # auc, accuracy, f1

vista_rest = {
    "name": r_info.json()["name"],
    "version": r_info.json()["model_version"],
    "algorithm": r_info.json()["algorithm"],
    "metrics": r_metrics.json(),
}
vista_rest

{'name': 'iris-rf',
 'version': '1.0.0',
 'algorithm': 'RandomForestClassifier',
 'metrics': {'accuracy': 0.9, 'auc': 0.9933, 'f1': 0.8997}}

## 6. Medición: llamadas y bytes

In [5]:
rest_calls = 2
rest_bytes = len(r_info.content) + len(r_metrics.content)

gql_calls = 1
gql_bytes = len(rg.content)

info = r_info.json()
campos_traidos = len(info)
campos_usados = 3   # name, version, algorithm

print(f"{'':12} {'llamadas':>10} {'bytes':>10}")
print(f"{'REST':12} {rest_calls:>10} {rest_bytes:>10}")
print(f"{'GraphQL':12} {gql_calls:>10} {gql_bytes:>10}")
print()
print(f"REST /v1/model: {campos_traidos} campos traídos, {campos_usados} usados "
      f"-> over-fetching de {campos_traidos - campos_usados} campos")
print(f"bytes: REST/{gql_bytes and round(rest_bytes/gql_bytes,1)}x  el de GraphQL")

               llamadas      bytes
REST                  2        341
GraphQL               1        158

REST /v1/model: 7 campos traídos, 3 usados -> over-fetching de 4 campos
bytes: REST/2.2x  el de GraphQL


## 7. Conclusión — REST vs GraphQL

|  | Llamadas | Campos de más |
|---|---|---|
| **REST** | 2 (`/v1/model` + `/v1/model/metrics`) | sí: `/v1/model` trae `dataset`, `features`, `target_names`, `trained_at` |
| **GraphQL** | 1 | no |

- **Under-fetching en REST:** ningún endpoint devuelve *la vista* completa, así
  que el cliente **compone** varias llamadas (2 acá; con recursos anidados
  aparecería el problema **N+1**).
- **Over-fetching en REST:** `/v1/model` devuelve todos los metadatos aunque la
  vista sólo use `name`, `version` y `algorithm`.
- **GraphQL:** una request con la forma exacta del dato. El cliente declara qué
  campos quiere y el servidor responde eso y nada más.
- **Costo de GraphQL:** hay que definir esquema y resolvers (más código y una
  capa conceptual extra). Para una API chica y estable, REST alcanza; GraphQL
  gana con muchos consumidores de necesidades distintas o grafos de datos con
  relaciones (linaje, MLflow).